In [4]:
import sys

sys.path.append("/Users/liu.9639/Downloads/A2")

import torch
import matplotlib.pyplot as plt
from a2.models.transformer_seq2seq import *
from a2.data import *
# Import established script functions and configs from your repository
from train_transformer import *

from train_mlp import *


"--ffn-dim","1024",
"--num-layers","4",
Test  | loss 0.1025 acc 0.8615

"--ffn-dim","1024",
"--num-layers","8",
Test  | loss 0.0485 acc 0.9406

"--ffn-dim","256",
"--num-layers","4",
Test  | loss 0.1063 acc 0.8683

"--ffn-dim","256",
"--nhead","32",
"--num-layers","4",
Test  | loss 0.0682 acc 0.9142

"--ffn-dim","256",
"--nhead","4",
"--num-layers","4",
Test  | loss 0.0446 acc 0.9472

"--ffn-dim","256",
"--nhead","8",
"--num-layers","4",
Test  | loss 0.0644 acc 0.9264

"--ffn-dim","256",
"--nhead","16",
"--num-layers","4",
Test  | loss 0.1063 acc 0.8683

"--ffn-dim","256",
"--nhead","4",
"--num-layers","2",
Test  | loss 0.4437 acc 0.3998

"--ffn-dim","256",
"--nhead",4",
"--num-layers","6",
Test  | loss 4.7074 acc 0.0000                                                               
Epoch 09 | train loss 0.0205 acc 0.9735 (full-set EM 0.9844) | val loss 0.0172 acc 0.9824                                                                                  
Epoch 10 | train loss 4.1637 acc 0.5891 (full-set EM 0.0009) | val loss 4.6122 acc 0.0015

"--ffn-dim","256",
"--nhead","4",
"--num-layers","8",                                                                           
Epoch 09 | train loss 0.0301 acc 0.9604 (full-set EM 0.9764) | val loss 0.0196 acc 0.9737                                                                                 
Epoch 10 | train loss 0.4446 acc 0.9149 (full-set EM 0.0009) | val loss 8.0916 acc 0.0015
Test  | loss 8.0414 acc 0.0013

"--ffn-dim","256",
"--nhead","4",
"--num-layers","10",
Test  | loss 0.0314 acc 0.9606

"--ffn-dim","256",
"--nhead","4",
"--num-layers","12",
Test  | loss 0.0626 acc 0.9219

"--ffn-dim","256",
"--nhead","4",
"--num-layers","14",
Test  | loss 0.0361 acc 0.9564

"--ffn-dim","256",
"--nhead","4",
"--num-layers","16",
Test  | loss 0.0488 acc 0.9378

"--ffn-dim","256",
"--nhead","4",
"--num-layers","18",
Test  | loss 0.0449 acc 0.9416

"--ffn-dim","256",
"--nhead","4",
"--num-layers","24",
Test  | loss 0.0408 acc 0.9491

"--ffn-dim","64",
"--nhead","4",
"--num-layers","4",
Test  | loss 0.1685 acc 0.7964

"--ffn-dim","128",
"--nhead","4",
"--num-layers","4",
Test  | loss 0.0955 acc 0.8738

"--ffn-dim","512",
"--nhead","4",
"--num-layers","4",
Test  | loss 0.0694 acc 0.9133

"--ffn-dim","1024",
"--nhead","4",
"--num-layers","4",
Test  | loss 0.0814 acc 0.9111

"--ffn-dim","256",
"--nhead","4",
"--num-layers","2",
"--epochs", "20"
Test  | loss 0.1054 acc 0.8858

"test set size=5000"
"--ffn-dim","256",
"--nhead","4",
"--num-layers","4",
Test  | loss 1.7358 acc 0.0029

In [5]:
import argparse

# Define the parser
parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=10)
parser.add_argument("--batch-size", type=int, default=128)
parser.add_argument("--d-model", type=int, default=128)
parser.add_argument("--nhead", type=int, default=4)  # ensure d_model % nhead == 0
parser.add_argument("--num-layers", type=int, default=2)
parser.add_argument("--ffn-dim", type=int, default=256)
parser.add_argument(
    "--dropout",
    type=float,
    default=0.1,
    help="Ignored by simplified model (no dropout implemented); kept for CLI compatibility",
)
parser.add_argument("--lr", type=float, default=1e-3)
parser.add_argument("--max-digits", type=int, default=3)
parser.add_argument("--test-max-digits", type=int, default=None, help="Max digits for the test set (default: --max-digits); set larger to test generalization to longer numbers")
parser.add_argument("--train-size", type=int, default=20000)
parser.add_argument("--val-size", type=int, default=2000)
parser.add_argument("--test-size", type=int, default=2000)
parser.add_argument("--weight-decay", type=float, default=0.01, help="AdamW weight decay")
parser.add_argument("--seed", type=int, default=42, help="Seed for model init / batch order")
parser.add_argument("--data-seed", type=int, default=0, help="Seed for the train/val/test data (and the label permutation)")
parser.add_argument("--random-labels", action="store_true", help="Shuffle targets across the training set (Zhang et al. 2017)")

args_list = [
    "--epochs", "5",
    "--max-digits", "2",
    "--train-size", "20000",
    "--val-size", "2000",
    "--test-size", "2000",
    "--ffn-dim","256",
    "--nhead","4",
    "--num-layers","4",
    "--test-max-digits", "3",
]


# Parse the argument list
args = parser.parse_args(args_list)

# Access your arguments as usual
print(args)

Namespace(epochs=5, batch_size=128, d_model=128, nhead=4, num_layers=4, ffn_dim=256, dropout=0.1, lr=0.001, max_digits=2, test_max_digits=3, train_size=20000, val_size=2000, test_size=2000, weight_decay=0.01, seed=42, data_seed=0, random_labels=False)


In [6]:

set_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = default_vocab()

train_ex = generate_dataset(args.train_size, args.max_digits, seed=args.data_seed)
val_ex = generate_dataset(args.val_size, args.max_digits, seed=args.data_seed + 1)
test_max_digits = args.test_max_digits or args.max_digits
test_ex = generate_dataset(args.test_size, test_max_digits, seed=args.data_seed + 2)
if args.random_labels:
    train_ex = permute_targets(train_ex, seed=args.data_seed)
    print("Training targets randomly permuted (--random-labels); val/test are unchanged.")

train_ds = AdditionDataset(train_ex, vocab)
val_ds = AdditionDataset(val_ex, vocab)
test_ds = AdditionDataset(test_ex, vocab)

collate = lambda b: collate_fn(b, pad_id=vocab.pad_id)  # noqa: E731
train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate)
train_eval_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)
val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False, collate_fn=collate)

print(f"Datasets: train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
print(f"Batches: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}")

model = TransformerSeq2Seq(
    TransformerConfig(
        vocab_size=vocab.size,
        d_model=args.d_model,
        nhead=args.nhead,
        num_layers=args.num_layers,
        dim_feedforward=args.ffn_dim,
        dropout=args.dropout,
        max_len=128,
    ),
    pad_id=vocab.pad_id,
    sos_id=vocab.sos_id,
    eos_id=vocab.eos_id,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)

best_val = 0.0
for epoch in range(1, args.epochs + 1):
    print(f"\nEpoch {epoch}/{args.epochs}")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
    # train_acc above is a running average over minibatches *during* the epoch; re-evaluate
    # the final weights on the whole training set for a clean train exact-match number.
    _, train_em = evaluate(model, train_eval_loader, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} (full-set EM {train_em:.4f}) | val loss {val_loss:.4f} acc {val_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test  | loss {test_loss:.4f} acc {test_acc:.4f}")



Datasets: train=20000 val=2000 test=2000
Batches: train=157 val=16 test=16

Epoch 1/5


Epoch 01 | train loss 5.7503 acc 0.0123 (full-set EM 0.0244) | val loss 1.5310 acc 0.0253

Epoch 2/5


Epoch 02 | train loss 1.4055 acc 0.0312 (full-set EM 0.0387) | val loss 1.3535 acc 0.0320

Epoch 3/5


Epoch 03 | train loss 1.1379 acc 0.1041 (full-set EM 0.2008) | val loss 0.9753 acc 0.1739

Epoch 4/5


Epoch 04 | train loss 0.6183 acc 0.3934 (full-set EM 0.6845) | val loss 0.2798 acc 0.6670

Epoch 5/5


Epoch 05 | train loss 0.1590 acc 0.8019 (full-set EM 0.8744) | val loss 0.1248 acc 0.8425


Test  | loss 27.2889 acc 0.0000
